In [1]:
# Import required libraries
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
from scipy import stats
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path

# Set publication-quality style
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 11

print("Libraries imported successfully")

Libraries imported successfully


## 1. Define Paths and Parameters

In [2]:
# Base directory
base_dir = Path('/home/renatob/data/FluoData1/aviris_dangermond')
output_dir = base_dir / 'shift_dangermond_trait' / 'figures'
output_dir.mkdir(parents=True, exist_ok=True)

# Dates for analysis (13 AVIRIS flights)
dates = [
    "2022-02-24T00:00:00.000000", "2022-02-28T00:00:00.000000", "2022-03-08T00:00:00.000000",
    "2022-03-16T00:00:00.000000", "2022-03-22T00:00:00.000000", "2022-04-05T00:00:00.000000",
    "2022-04-12T00:00:00.000000", "2022-04-20T00:00:00.000000", "2022-04-29T00:00:00.000000",
    "2022-05-03T00:00:00.000000", "2022-05-11T00:00:00.000000", "2022-05-17T00:00:00.000000",
    "2022-05-29T00:00:00.000000"
]

# File paths (corrected to use local paths)
base_file_path_trait = base_dir / 'fitting' / 'fitted_prescribed_lai_ci' / 'shift_fluxes_day_{:02d}_clima_fit_reg_jmax.nc'
base_file_path_pft = base_dir / 'fitting' / 'fitted_prescribed_lai_ci' / 'pft_shift_fluxes_day_{:02d}_clima_fit_reg_jmax.nc'
pft_file = base_dir / 'California_Vegetation_WHRTYPE_Dangermond' / 'output_latlon.nc'

print(f"Processing {len(dates)} dates")
print(f"Output directory: {output_dir}")

Processing 13 dates
Output directory: /home/renatob/data/FluoData1/aviris_dangermond/shift_dangermond_trait/figures


## 2. Load PFT Map and Calculate Spatial Means

In [3]:
# Load PFT map
print("Loading PFT map...")
pft_ds = xr.open_dataset(pft_file)
pft_map = pft_ds['Band1'].values  # Variable is 'Band1' not 'pft'

print(f"PFT map shape: {pft_map.shape}")
print(f"Unique PFT values: {np.unique(pft_map[~np.isnan(pft_map)])}")

# Get coordinates from first flux file
first_file = str(base_file_path_trait).format(0)
with xr.open_dataset(first_file, decode_times=False) as ds:
    lats = ds['lat'].values
    lons = ds['lon'].values
    
print(f"Coordinate ranges: Lat [{lats.min():.4f}, {lats.max():.4f}], Lon [{lons.min():.4f}, {lons.max():.4f}]")

Loading PFT map...
PFT map shape: (458, 492)
Unique PFT values: [1. 2. 3. 4. 5. 6. 7.]
Coordinate ranges: Lat [34.4413, 34.5776], Lon [-120.5019, -120.3555]


In [4]:
# Initialize arrays for spatial means
gpp_diff_spatial = None
sif_diff_spatial = None

print("Processing flux files...")
# Process each date
for i, date in enumerate(dates):
    file_trait = str(base_file_path_trait).format(i)
    file_pft = str(base_file_path_pft).format(i)
    
    # Open datasets with decode_times=False to avoid datetime issues
    ds_trait = xr.open_dataset(file_trait, decode_times=False)
    ds_pft = xr.open_dataset(file_pft, decode_times=False)
    
    # Extract GPP and SIF
    gpp_trait = ds_trait['gpp'].values
    gpp_pft = ds_pft['gpp'].values
    sif_trait = ds_trait['sif740'].values  # Note: variable name may be 'sif' or 'sif740'
    sif_pft = ds_pft['sif740'].values
    
    # Calculate differences and accumulate
    if gpp_diff_spatial is None:
        gpp_diff_spatial = gpp_trait - gpp_pft
        sif_diff_spatial = sif_trait - sif_pft
    else:
        gpp_diff_spatial += gpp_trait - gpp_pft
        sif_diff_spatial += sif_trait - sif_pft
    
    ds_trait.close()
    ds_pft.close()
    
    if (i + 1) % 5 == 0:
        print(f"  Processed {i + 1}/{len(dates)} dates")

# Calculate temporal averages
gpp_diff_spatial /= len(dates)
sif_diff_spatial /= len(dates)

print(f"\nSpatial mean GPP difference: {np.nanmean(gpp_diff_spatial):.3f} ± {np.nanstd(gpp_diff_spatial):.3f} µmol/m²/s")
print(f"Spatial mean SIF difference: {np.nanmean(sif_diff_spatial):.4f} ± {np.nanstd(sif_diff_spatial):.4f} mW/m²/nm/sr")

Processing flux files...
  Processed 5/13 dates
  Processed 10/13 dates

Spatial mean GPP difference: 0.046 ± 2.643 µmol/m²/s
Spatial mean SIF difference: 0.0032 ± 0.0299 mW/m²/nm/sr


## 3. Define Plotting Functions

In [5]:
def add_density_inset_pft(ax, data, pft_map, variable_name, position='lower right'):
    """
    Add density distribution inset aggregated per PFT.
    
    Parameters:
    -----------
    ax : matplotlib axis
        The axis to add the inset to
    data : numpy array
        2D spatial data array
    pft_map : numpy array
        2D PFT classification map
    variable_name : str
        Name of the variable for labeling
    position : str
        Position of inset ('lower right', 'upper right', etc.)
    """
    # Create inset axes (35% size for better visibility)
    if position == 'lower right':
        axins = inset_axes(ax, width="35%", height="35%", loc='lower right',
                          bbox_to_anchor=(0.05, 0.05, 0.9, 0.9), bbox_transform=ax.transAxes,
                          borderpad=0)
    else:
        axins = inset_axes(ax, width="35%", height="35%", loc=position, borderpad=1.0)
    
    # Define PFT colors (matching common vegetation types)
    pft_colors = {2: 'blue', 3: 'green', 4: 'orange'}  # PFTs 2, 3, 4 at Dangermond
    pft_names = {2: 'PFT 2', 3: 'PFT 3', 4: 'PFT 4'}
    
    # Flatten data and remove NaNs
    flat_data = data.flatten()
    flat_pft = pft_map.flatten()
    
    valid_mask = ~np.isnan(flat_data) & ~np.isnan(flat_pft)
    flat_data = flat_data[valid_mask]
    flat_pft = flat_pft[valid_mask]
    
    # Calculate overall statistics
    data_mean = np.mean(flat_data)
    data_std = np.std(flat_data)
    
    # Plot density for each PFT
    for pft_id in [2, 3, 4]:  # Dangermond has PFTs 2, 3, 4
        pft_data = flat_data[flat_pft == pft_id]
        if len(pft_data) > 0:
            axins.hist(pft_data, bins=40, alpha=0.6, density=True, 
                      color=pft_colors[pft_id], label=pft_names[pft_id],
                      edgecolor='black', linewidth=0.5)
    
    # Add reference lines
    axins.axvline(0, color='red', linestyle='--', linewidth=1.5, alpha=0.8, label='Zero')
    axins.axvline(data_mean, color='black', linestyle='--', linewidth=1.5, alpha=0.8, label='Mean')
    
    # Styling
    axins.set_xlabel(f'Δ{variable_name}', fontsize=10, fontweight='bold')
    axins.set_ylabel('Density', fontsize=10)
    axins.tick_params(labelsize=8)
    axins.legend(fontsize=7, loc='upper right', framealpha=0.9)
    axins.grid(True, alpha=0.3, linewidth=0.5)
    axins.set_facecolor('white')
    axins.patch.set_alpha(0.95)
    
    # Add border
    for spine in axins.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(1.5)
    
    return axins

print("Plotting functions defined")

Plotting functions defined


## 4. Create Figure 6 - Spatial Maps Only (Panels a & b)

Note: Temporal panels (c & d) will be merged with Figure 7 as per reviewer suggestion

In [11]:
# Squeeze out singleton time dimension to prepare for plotting
print(f"Original shapes: gpp_diff_spatial={gpp_diff_spatial.shape}, sif_diff_spatial={sif_diff_spatial.shape}")
gpp_diff_spatial = gpp_diff_spatial.squeeze()
sif_diff_spatial = sif_diff_spatial.squeeze()
print(f"After squeeze: gpp_diff_spatial={gpp_diff_spatial.shape}, sif_diff_spatial={sif_diff_spatial.shape}")
print("Ready for plotting!")

Original shapes: gpp_diff_spatial=(458, 492), sif_diff_spatial=(458, 492)
After squeeze: gpp_diff_spatial=(458, 492), sif_diff_spatial=(458, 492)
Ready for plotting!


In [13]:
# Create figure with 1x2 layout (only spatial maps)
fig, axes = plt.subplots(1, 2, figsize=(18, 8), dpi=150)

# Create meshgrid for plotting
LON, LAT = np.meshgrid(lons, lats)

# Panel a: GPP difference spatial map
ax1 = axes[0]
im1 = ax1.pcolormesh(LON, LAT, gpp_diff_spatial, cmap='RdBu_r',
                     vmin=-5, vmax=5, shading='auto', rasterized=True)
ax1.set_aspect('equal')

# Add coastlines for geographic context
#ax1.coastlines(resolution='10m', linewidth=0.8, alpha=0.7, color='black')

# Panel label
ax1.text(0.02, 0.98, '(a)', transform=ax1.transAxes, fontsize=22, fontweight='bold',
         va='top', ha='left', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='black'))

# Add statistics box
gpp_mean = np.nanmean(gpp_diff_spatial)
gpp_std = np.nanstd(gpp_diff_spatial)
stats_text = f'Mean: {gpp_mean:.2f}\nSTD: {gpp_std:.2f}'
ax1.text(0.98, 0.98, stats_text, transform=ax1.transAxes, fontsize=14,
         va='top', ha='right', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='black'))

# Colorbar
cbar1 = plt.colorbar(im1, ax=ax1, orientation='vertical', pad=0.02, fraction=0.046)
cbar1.set_label(r'GPP Difference (µmol CO$_2$ m$^{-2}$ s$^{-1}$)', fontsize=16, fontweight='bold')
cbar1.ax.tick_params(labelsize=13)

# Add density inset aggregated per PFT
add_density_inset_pft(ax1, gpp_diff_spatial, pft_map, 'GPP', position='lower right')

# Axis labels
ax1.set_xlabel('Longitude', fontsize=15, fontweight='bold')
ax1.set_ylabel('Latitude', fontsize=15, fontweight='bold')
ax1.tick_params(labelsize=12)

# Panel b: SIF difference spatial map
ax2 = axes[1]
im2 = ax2.pcolormesh(LON, LAT, sif_diff_spatial, cmap='RdBu_r',
                     vmin=-0.25, vmax=0.25, shading='auto', rasterized=True)
ax2.set_aspect('equal')

# Add coastlines for geographic context
#ax2.coastlines(resolution='10m', linewidth=0.8, alpha=0.7, color='black')

# Panel label
ax2.text(0.02, 0.98, '(b)', transform=ax2.transAxes, fontsize=22, fontweight='bold',
         va='top', ha='left', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='black'))

# Add statistics box
sif_mean = np.nanmean(sif_diff_spatial)
sif_std = np.nanstd(sif_diff_spatial)
stats_text = f'Mean: {sif_mean:.3f}\nSTD: {sif_std:.3f}'
ax2.text(0.98, 0.98, stats_text, transform=ax2.transAxes, fontsize=14,
         va='top', ha='right', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='black'))

# Colorbar
cbar2 = plt.colorbar(im2, ax=ax2, orientation='vertical', pad=0.02, fraction=0.046)
cbar2.set_label(r'SIF$_{740nm}$ Difference (mW m$^{-2}$ sr$^{-1}$ nm$^{-1}$)', fontsize=16, fontweight='bold')
cbar2.ax.tick_params(labelsize=13)

# Add density inset aggregated per PFT
add_density_inset_pft(ax2, sif_diff_spatial, pft_map, 'SIF', position='lower right')

# Axis labels
ax2.set_xlabel('Longitude', fontsize=15, fontweight='bold')
ax2.set_ylabel('Latitude', fontsize=15, fontweight='bold')
ax2.tick_params(labelsize=12)

# Adjust layout
plt.tight_layout()

# Save figure
output_file = output_dir / 'figure_6_gpp_sif_spatial_revised.png'
plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
print(f"\nFigure saved to: {output_file}")

# Close to prevent rendering errors
plt.close(fig)

print("\n" + "="*80)
print("FIGURE 6 GENERATION COMPLETE")
print("="*80)
print("\n✅ Changes applied:")
print("  • Added density insets aggregated per PFT to panels a and b")
print("  • Removed temporal panels c and d (to be merged with Figure 7)")
print("  • Increased label sizes for publication quality")
print("  • Added coastlines for geographic context")
print("  • Statistics boxes show Mean/STD for each panel")

/tmp/ipykernel_4085088/4100654826.py:74: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


AttributeError: 'NoneType' object has no attribute '_get_renderer'

AttributeError: 'NoneType' object has no attribute '_get_renderer'

<Figure size 2700x1200 with 6 Axes>

## 5. Display Saved Figure

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(output_file), width=1400))

NameError: name 'output_file' is not defined

## 6. Summary Statistics

In [ ]:
# Calculate and print detailed summary statistics
print("=" * 80)
print("FIGURE 6 SUMMARY STATISTICS")
print("=" * 80)

# Overall spatial differences
gpp_diff_flat = gpp_diff_spatial.flatten()
gpp_diff_flat = gpp_diff_flat[~np.isnan(gpp_diff_flat)]
print(f"\nGPP Spatial Difference (Trait - PFT):")
print(f"  Mean: {np.mean(gpp_diff_flat):.3f} µmol CO₂ m⁻² s⁻¹")
print(f"  Std:  {np.std(gpp_diff_flat):.3f} µmol CO₂ m⁻² s⁻¹")
print(f"  Min:  {np.min(gpp_diff_flat):.3f} µmol CO₂ m⁻² s⁻¹")
print(f"  Max:  {np.max(gpp_diff_flat):.3f} µmol CO₂ m⁻² s⁻¹")

sif_diff_flat = sif_diff_spatial.flatten()
sif_diff_flat = sif_diff_flat[~np.isnan(sif_diff_flat)]
print(f"\nSIF Spatial Difference (Trait - PFT):")
print(f"  Mean: {np.mean(sif_diff_flat):.4f} mW m⁻² sr⁻¹ nm⁻¹")
print(f"  Std:  {np.std(sif_diff_flat):.4f} mW m⁻² sr⁻¹ nm⁻¹")
print(f"  Min:  {np.min(sif_diff_flat):.4f} mW m⁻² sr⁻¹ nm⁻¹")
print(f"  Max:  {np.max(sif_diff_flat):.4f} mW m⁻² sr⁻¹ nm⁻¹")

# Per-PFT statistics
flat_pft = pft_map.flatten()
valid_mask = ~np.isnan(gpp_diff_spatial.flatten()) & ~np.isnan(flat_pft)
gpp_diff_valid = gpp_diff_spatial.flatten()[valid_mask]
sif_diff_valid = sif_diff_spatial.flatten()[valid_mask]
pft_valid = flat_pft[valid_mask]

print("\n" + "="*80)
print("PER-PFT STATISTICS")
print("="*80)

for pft_id in [2, 3, 4]:
    pft_mask = pft_valid == pft_id
    n_pixels = pft_mask.sum()
    
    if n_pixels > 0:
        gpp_pft = gpp_diff_valid[pft_mask]
        sif_pft = sif_diff_valid[pft_mask]
        
        print(f"\nPFT {pft_id} (n={n_pixels} pixels):")
        print(f"  GPP: Mean={np.mean(gpp_pft):.3f}, Std={np.std(gpp_pft):.3f}, Range=[{np.min(gpp_pft):.3f}, {np.max(gpp_pft):.3f}]")
        print(f"  SIF: Mean={np.mean(sif_pft):.4f}, Std={np.std(sif_pft):.4f}, Range=[{np.min(sif_pft):.4f}, {np.max(sif_pft):.4f}]")

print("\n" + "="*80)

## 7. Export High-Resolution Version

In [ ]:
# Create high-resolution version for publication
print("Creating high-resolution version (600 DPI)...")

fig_hires, axes_hires = plt.subplots(1, 2, figsize=(18, 8), dpi=600)

# Replicate the main figure
LON, LAT = np.meshgrid(lons, lats)

# Panel a: GPP
ax1 = axes_hires[0]
im1 = ax1.pcolormesh(LON, LAT, gpp_diff_spatial, cmap='RdBu_r',
                     vmin=-5, vmax=5, shading='auto', rasterized=True)
ax1.set_aspect('equal')
ax1.coastlines(resolution='10m', linewidth=0.8, alpha=0.7, color='black')
ax1.text(0.02, 0.98, '(a)', transform=ax1.transAxes, fontsize=22, fontweight='bold',
         va='top', ha='left', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='black'))
ax1.text(0.98, 0.98, f'Mean: {gpp_mean:.2f}\nSTD: {gpp_std:.2f}', transform=ax1.transAxes,
         fontsize=14, va='top', ha='right', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='black'))
cbar1 = plt.colorbar(im1, ax=ax1, orientation='vertical', pad=0.02, fraction=0.046)
cbar1.set_label(r'GPP Difference (µmol CO$_2$ m$^{-2}$ s$^{-1}$)', fontsize=16, fontweight='bold')
cbar1.ax.tick_params(labelsize=13)
add_density_inset_pft(ax1, gpp_diff_spatial, pft_map, 'GPP', position='lower right')
ax1.set_xlabel('Longitude', fontsize=15, fontweight='bold')
ax1.set_ylabel('Latitude', fontsize=15, fontweight='bold')
ax1.tick_params(labelsize=12)

# Panel b: SIF
ax2 = axes_hires[1]
im2 = ax2.pcolormesh(LON, LAT, sif_diff_spatial, cmap='RdBu_r',
                     vmin=-0.25, vmax=0.25, shading='auto', rasterized=True)
ax2.set_aspect('equal')
ax2.coastlines(resolution='10m', linewidth=0.8, alpha=0.7, color='black')
ax2.text(0.02, 0.98, '(b)', transform=ax2.transAxes, fontsize=22, fontweight='bold',
         va='top', ha='left', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='black'))
ax2.text(0.98, 0.98, f'Mean: {sif_mean:.3f}\nSTD: {sif_std:.3f}', transform=ax2.transAxes,
         fontsize=14, va='top', ha='right', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='black'))
cbar2 = plt.colorbar(im2, ax=ax2, orientation='vertical', pad=0.02, fraction=0.046)
cbar2.set_label(r'SIF$_{740nm}$ Difference (mW m$^{-2}$ sr$^{-1}$ nm$^{-1}$)', fontsize=16, fontweight='bold')
cbar2.ax.tick_params(labelsize=13)
add_density_inset_pft(ax2, sif_diff_spatial, pft_map, 'SIF', position='lower right')
ax2.set_xlabel('Longitude', fontsize=15, fontweight='bold')
ax2.set_ylabel('Latitude', fontsize=15, fontweight='bold')
ax2.tick_params(labelsize=12)

plt.tight_layout()

# Save high-resolution versions
output_hires_png = output_dir / 'figure_6_gpp_sif_spatial_600dpi.png'
output_hires_pdf = output_dir / 'figure_6_gpp_sif_spatial.pdf'

plt.savefig(output_hires_png, dpi=600, bbox_inches='tight', facecolor='white')
plt.savefig(output_hires_pdf, format='pdf', bbox_inches='tight', facecolor='white')

print(f"\nHigh-resolution versions saved:")
print(f"  PNG (600 DPI): {output_hires_png}")
print(f"  PDF: {output_hires_pdf}")

plt.close(fig_hires)

print("\n" + "="*80)
print("ALL VERSIONS GENERATED SUCCESSFULLY!")
print("="*80)